In [ ]:
# learn_data_suplit.pyを参考に作成
# learn_data_suplit.pyでは，最後の部分モデルに信号線の余りを全て入れていたが，learn_data_suplit4.pyでは，余りを各部分モデルに1つずつ分配するように変更した，
# この時，ノード数を増やす部分モデルは，最後の部分モデルから数えて余りの個数分となる．（例　モデルが5つ，余りが3の場合，最後のモデルとその一つ前のモデル,もうひとつ前部分モデルの出力ノード数を1つずつ増やす）
# 正解データを分割するプログラム
# 正解データを0~1に正規化. 0~3の値を0~1に変換。0->0, 1->0.375, 2->0.625, 3->1 (※値がなるべく閾値から離れるように0は0のまま、3は1とした)
# cd workspace/research2/experiment
#　実行コマンド：　python3 learn_data_suplit.py

#グローバル変数
cir = 's1494'  # 対象回路
suplit_num = 20 # 何個ずつ分割するか
correct_data_file = cir + 'integrated_output'  # 正解データファイル
correct_data_folder = cir + '分割正解データ2' # 正解データを分割して保存するフォルダ
suplit_correct_data_file = cir + 'integrated_output'  # 分割された正解データファイル
correct_data_value_file = cir + 'correct_value'  # 正解データの種類（値）を保存するファイル
threshold_file = cir + 'threshold'  # 閾値を保存するファイル
suplit_num_file = cir + 'suplit_num'  # 分割数を保存するファイル
model_output_node_num_file = cir + 'model_output_node_num'  # 部分モデルの出力ノード数を保存するファイル
single_line_file = cir + 'single_line' 
correct_value_count_file = correct_data_folder + "/" + cir + 'correct_value_count'  # 正解データの値の数を保存するファイル
correct_value = [0.00, 0.25, 0.75, 1.00]  # 正解データの値
threshold = [0.02, 0.5, 0.98]


# 正解データファイルを開いてデータを読み込む
with open(correct_data_file) as f:
  single_flag = int(f.readline().split()[0])  # 最適なペアが存在しない信号線があるかどうかを示す # 0:存在しない, 1:存在する

  # 2行目 統合後の信号線の数を読み込む(ペアが存在しない信号線の数も含む)
  signal_sum = int(f.readline())

  # 3行目（最適なペア信号線）を読み込む。カンマ区切りで各信号線をリストに格納
  best_pair = f.readline().split(',')

  # 4行目以降を読み込む。_にいれた各文字列の空白文字を削除
  correct_data = [_.replace(",", "").replace("\n", "") for _ in f.readlines()]  # 各行の空白文字と改行文字を削除

correct_data = [list(_) for _ in correct_data]  # 1次元リストを2次元リストに変換

print(len(correct_data))
# print(correct_data[0])
# print(correct_data[0][0])
# print(type(correct_data[0]))

print("signal_sum:", signal_sum)  # 統合された信号線の数

# モデルの分割数を計算
model_suplit_num = signal_sum // suplit_num  #　統合された信号線数を分割数で割る
print("model_suplit_num：", model_suplit_num)

# 分割された正解データを格納するリスト
suplit_correct_data = [["0" for _ in range(len(correct_data[0]))] for _ in range(suplit_num)] # 2次元配列を0で初期化。列数は故障候補値の数（correct_dataの列数）、行数はsuplit_num=分割数

remainder_flag = 0  # signal_sumをsuplit_numで割り切れる亜銅貨を示すフラグ
remainder_num = 0  # 割り切れない数を格納する変数
if signal_sum % suplit_num != 0:  # signal_sumがsuplit_numで割り切れない場合
    remainder_flag = 1 # 割り切れないフラグを立てる
    remainder_num = signal_sum % suplit_num  # 割り切れない数を求める

print("remainder_num:", remainder_num)
print("remainder_num//model_suplit_num:", remainder_num//model_suplit_num)

# 部分モデルの出力ノード数を計算
# 余りがある場合，余りの数remainder_numと同数の部分モデルの出力ノード数を1つずつ増やす．この時，ノード数を増やす部分モデルは，最後の部分モデルから数えてremainder_num個分となる．
output_node_num = [suplit_num] * model_suplit_num # 部分モデルの出力ノード数を格納するリスト．初期値はsuplit_numで初期化．余りがない場合は，このままの値を使用する．
if (remainder_num > model_suplit_num) and (remainder_flag == 1):  # 余りがあり，かつ，余りが部分モデルの数より大きい場合.全てのモデルの出力ノード数を1つずつ増やす
        for j in range(remainder_num//model_suplit_num):  # 余りの数が部分モデルの何倍かを計算し，倍数分だけ各部分モデルの出力ノード数を1つずつ増やす
            output_node_num = [x + 1 for x in output_node_num]
            remainder_num -= model_suplit_num

print("remainder_num:", remainder_num)
if (remainder_num > 0) and (remainder_flag == 1):  # 残りの部分モデル数が余り以下の場合、かつ、余りがある場合
    for i in range(model_suplit_num):
        output_node_num[model_suplit_num - 1 - i] += 1
        remainder_num -= 1
        if remainder_num == 0:
            break

print("output_node_num:", output_node_num)

correct_value_count = [[0, 0, 0, 0] for _ in range(model_suplit_num)]  # 正解データの値のカウント用リスト
suplit_sum_count = 0  # 分割された正解データの信号線数のカウント用変数
for i in range(model_suplit_num):  # モデルの分割数分繰り返す
    with open(correct_data_folder + '/' + suplit_correct_data_file + str(i), 'w') as f:  # 分割された正解データを書き込むファイルを開く
        for j in range(len(correct_data)):
            # print("output_node_num:", output_node_num)
            # print("i:", i)
            for k in range(suplit_sum_count, suplit_sum_count + output_node_num[i]):    # 分割数ごとに左から右にoutput_node_num個ずつ正解データを書き込む.正解データファイルごとに、部分モデルの出力ノード数（output_node_num）だけ、左にずれるので、suplit_sum_count変数で合計をカウントする。
                if (single_flag == 1) and (i == (model_suplit_num - 1)) and (k == (suplit_sum_count + output_node_num[i] - 1)):  # 最適なペアが存在しない信号線が存在し、ペアが存在しない信号線だった場合 = 最後の部分モデルの最後の要素の場合（つまり、ペアが存在しない信号線の値は書き替えない）
                    if correct_data[j][k] == '0':
                        correct_data[j][k] = correct_value[0]
                        correct_value_count[i][0] += 1
                    elif correct_data[j][k] == '1':
                        correct_data[j][k] = correct_value[3]
                        correct_value_count[i][3] += 1
                else:
                    # 正解データを0~1に正規化. 0~3の値を0~1に変換。
                    if correct_data[j][k] == '0':
                        correct_data[j][k] = correct_value[0]
                        correct_value_count[i][0] += 1
                    elif correct_data[j][k] == '1':
                        correct_data[j][k] = correct_value[1]
                        correct_value_count[i][1] += 1
                    elif correct_data[j][k] == '2':
                        correct_data[j][k] = correct_value[2]
                        correct_value_count[i][2] += 1
                    elif correct_data[j][k] == '3':
                        correct_data[j][k] = correct_value[3]
                        correct_value_count[i][3] += 1

                if k < suplit_sum_count + output_node_num[i] - 1:  # 最後の要素以外はカンマを付ける
                    f.write(str(correct_data[j][k]) + ',')
                else:                                  # 最後の要素はカンマを付けない
                    f.write(str(correct_data[j][k]) + '\n')
            
    suplit_sum_count += output_node_num[i]  # 分割された正解データの信号線数をカウント
    

with open(correct_value_count_file, 'w') as f:
    for i in range(model_suplit_num):
        for j in range(len(correct_value)):
            f.write("モデル" + str(i+1) + ": " + str(correct_value_count[i][j]) + '\n')
        f.write('\n')


sum_correct_value_count = [sum(col) for col in zip(*correct_value_count)]  # 各正解データの値の合計を計算
for i in range(len(correct_value)):
    print(f'正解データの値 {correct_value[i]} の数: {sum_correct_value_count[i]}')  # 正解データの値の数を表示

# 正解データの値を保存するファイルを開く
with open(correct_data_folder + '/' + correct_data_value_file, 'w') as f:  # 正解データの値を保存するファイルを開く
    for i in range(len(correct_value)):
        f.write(str(correct_value[i]) + ' ')
    f.write('\n')  # 正解データの値を保存する

# 閾値を保存するファイルを開く
with open(correct_data_folder + '/' + threshold_file, 'w') as f:  #
    for i in range(len(threshold)):
        f.write(str(threshold[i]) + ' ')
    f.write('\n')  # 閾値を保存する

with open(correct_data_folder + '/' + cir + 'suplit_data_num', 'w') as f:  # データの分割数を保存するファイルを開く
    for i in range(len(output_node_num)):
        f.write(str(output_node_num[i]) + '\n')  # データの分割数を保存する

with open(correct_data_folder + '/' + suplit_num_file, 'w') as f: # モデルの分割数を保存するファイルを開く
    f.write(str(model_suplit_num) + '\n')

# 統合されていない信号線があるかどうかを保存 0:存在しない、1:存在する
with open(correct_data_folder + '/' + single_line_file, 'w') as f:
    f.write(str(single_flag) + '\n')
    if single_flag == 0:  # 最適なペアが存在しない信号線が存在する場合
        f.write('統合されていない信号線はありません。\n')
    else:  # 最適なペアが存在しない信号線が存在しない場合
        f.write('統合されていない信号線があります。\n')
        

1848
signal_sum: 747
model_suplit_num： 37
remainder_num: 7
remainder_num//model_suplit_num: 0
remainder_num: 7
output_node_num: [20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 21, 21, 21, 21, 21, 21, 21]
正解データの値 0.0 の数: 1296910
正解データの値 0.25 の数: 51628
正解データの値 0.75 の数: 9488
正解データの値 1.0 の数: 22430


In [11]:
output_node_num = [suplit_num] * model_suplit_num # 部分モデルの出力ノード数を格納するリスト
print("output_node_num:", output_node_num)

output_node_num: [30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30]
